In [2]:
!pip -q install requests beautifulsoup4 pandas rapidfuzz playwright


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 30.9 MB/s eta 0:00:00


In [5]:
from __future__ import annotations

import re
import os
import csv
import gzip
import time
import html as html_lib
from pathlib import Path
from datetime import datetime, timedelta, date
from typing import Dict, List, Optional, Tuple
from urllib.parse import urljoin, quote, urlparse, parse_qs, unquote

import requests
import pandas as pd
from bs4 import BeautifulSoup, Tag
from rapidfuzz import fuzz
from IPython.display import display, HTML


# =========================
# CONFIG
# =========================
AMC_BASE = "https://www.amctheatres.com"
RT_BASE  = "https://www.rottentomatoes.com"

THEATRES = [
    {"name": "AMC Tustin 14 @ The District",
     "url": "https://www.amctheatres.com/movie-theatres/los-angeles/amc-tustin-14-at-the-district/showtimes"},
    {"name": "AMC Orange 30",
     "url": "https://www.amctheatres.com/movie-theatres/los-angeles/amc-orange-30/showtimes"},
    {"name": "AMC Woodbridge 5",
     "url": "https://www.amctheatres.com/movie-theatres/los-angeles/amc-woodbridge-5/showtimes"},
]

# Optional: pin a specific weekend Saturday (YYYY-MM-DD) for testing
OVERRIDE_SATURDAY = None  # e.g. "2025-12-20"

# Turn on to see RT URL attempts
DEBUG_RT = False

IMDB_DATASET_DIR = Path("build/imdb_datasets")
IMDB_BASICS_GZ = IMDB_DATASET_DIR / "title.basics.tsv.gz"
IMDB_RATINGS_GZ = IMDB_DATASET_DIR / "title.ratings.tsv.gz"
IMDB_BASICS_URL = "https://datasets.imdbws.com/title.basics.tsv.gz"
IMDB_RATINGS_URL = "https://datasets.imdbws.com/title.ratings.tsv.gz"


def safe_html_attr(value: object) -> str:
    return html_lib.escape(str(value or ''), quote=True)


def link_html(url: Optional[str], text: object) -> str:
    label = '' if text is None else str(text)
    if isinstance(url, str) and url.startswith('http'):
        return f'<a href="{safe_html_attr(url)}" target="_blank" rel="noopener noreferrer">{html_lib.escape(label)}</a>'
    return html_lib.escape(label)


def normalize_title_for_match(title: str) -> str:
    s = (title or '').lower().strip()
    s = re.sub(r'[^a-z0-9]+', ' ', s)
    s = re.sub(r'\b(the|a|an)\b', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s


def _clean_missing_text(value: object) -> str:
    s = '' if value is None else str(value).strip()
    return '' if s.lower() in {'', 'none', 'nan', 'null', '\n'} else s


def _safe_int(value: object) -> Optional[int]:
    s = _clean_missing_text(value)
    if not s:
        return None
    try:
        return int(s)
    except Exception:
        return None


def _safe_float(value: object) -> Optional[float]:
    s = _clean_missing_text(value)
    if not s:
        return None
    try:
        return float(s)
    except Exception:
        return None


def _download_to_path(session: requests.Session, url: str, dest: Path, tries: int = 3) -> Path:
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix(dest.suffix + '.part')
    last_err = None
    for attempt in range(tries):
        try:
            with session.get(url, stream=True, timeout=180, allow_redirects=True) as r:
                r.raise_for_status()
                with tmp.open('wb') as f:
                    for chunk in r.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            f.write(chunk)
            tmp.replace(dest)
            return dest
        except Exception as e:
            last_err = e
            try:
                tmp.unlink(missing_ok=True)
            except Exception:
                pass
            time.sleep(attempt + 1)
    raise RuntimeError(f'Failed downloading {url}: {last_err}')


def search_result_urls(session: requests.Session, query: str, allowed_prefixes: tuple[str, ...], max_results: int = 8) -> List[str]:
    urls: List[str] = []
    seen = set()
    endpoints = [
        ('https://html.duckduckgo.com/html/', {'q': query}),
        ('https://duckduckgo.com/html/', {'q': query}),
    ]
    for endpoint, params in endpoints:
        try:
            r = session.get(endpoint, params=params, timeout=30, allow_redirects=True)
            r.raise_for_status()
        except Exception:
            continue

        soup = BeautifulSoup(r.text or '', 'html.parser')
        for a in soup.find_all('a', href=True):
            href = (a.get('href') or '').strip()
            if not href:
                continue
            resolved = href
            if 'duckduckgo.com/l/?' in href or href.startswith('/l/?'):
                parsed = urlparse(href)
                qs = parse_qs(parsed.query)
                resolved = unquote((qs.get('uddg') or [''])[0])
            if not resolved.startswith('http'):
                continue
            if not any(resolved.startswith(p) for p in allowed_prefixes):
                continue
            clean = resolved.split('?', 1)[0]
            if clean not in seen:
                urls.append(clean)
                seen.add(clean)
            if len(urls) >= max_results:
                return urls
    return urls


def build_imdb_lookup(session: requests.Session, titles: List[str]) -> Dict[str, Tuple[Optional[float], Optional[int], Optional[str]]]:
    title_variants: Dict[str, List[str]] = {}
    variant_to_titles: Dict[str, set[str]] = {}
    for title in titles:
        norms: List[str] = []
        seen = set()
        for variant in candidate_title_variants(title):
            norm = normalize_title_for_match(variant)
            if norm and norm not in seen:
                norms.append(norm)
                seen.add(norm)
                variant_to_titles.setdefault(norm, set()).add(title)
        title_variants[title] = norms

    _download_to_path(session, IMDB_BASICS_URL, IMDB_BASICS_GZ)
    _download_to_path(session, IMDB_RATINGS_URL, IMDB_RATINGS_GZ)

    candidate_map: Dict[str, List[dict]] = {title: [] for title in titles}
    with gzip.open(IMDB_BASICS_GZ, 'rt', encoding='utf-8', newline='') as f:
        reader = csv.DictReader(f, delimiter='\t')
        for row in reader:
            if row.get('titleType') not in {'movie', 'tvMovie'}:
                continue
            if row.get('isAdult') == '1':
                continue

            primary_norm = normalize_title_for_match(row.get('primaryTitle', ''))
            original_norm = normalize_title_for_match(row.get('originalTitle', ''))
            matched_titles = set()
            if primary_norm:
                matched_titles |= variant_to_titles.get(primary_norm, set())
            if original_norm:
                matched_titles |= variant_to_titles.get(original_norm, set())
            if not matched_titles:
                continue

            cand = {
                'tconst': row.get('tconst', ''),
                'titleType': row.get('titleType', ''),
                'primaryTitle': row.get('primaryTitle', ''),
                'originalTitle': row.get('originalTitle', ''),
                'primaryNorm': primary_norm,
                'originalNorm': original_norm,
                'startYear': _safe_int(row.get('startYear')),
                'runtimeMinutes': _safe_int(row.get('runtimeMinutes')),
            }
            for title in matched_titles:
                candidate_map[title].append(cand)

    if not any(candidate_map.values()):
        return {title: (None, None, None) for title in titles}

    rating_rows: Dict[str, Tuple[Optional[float], int]] = {}
    wanted_ids = {cand['tconst'] for cands in candidate_map.values() for cand in cands if cand.get('tconst')}
    with gzip.open(IMDB_RATINGS_GZ, 'rt', encoding='utf-8', newline='') as f:
        reader = csv.DictReader(f, delimiter='\t')
        for row in reader:
            tconst = row.get('tconst', '')
            if tconst not in wanted_ids:
                continue
            rating_rows[tconst] = (_safe_float(row.get('averageRating')), _safe_int(row.get('numVotes')) or 0)

    current_year = date.today().year
    out: Dict[str, Tuple[Optional[float], Optional[int], Optional[str]]] = {}
    for title in titles:
        target = normalize_title_for_match(title)
        chosen = None
        chosen_key = None
        for cand in candidate_map.get(title, []):
            rating_val, votes = rating_rows.get(cand['tconst'], (None, 0))
            exact = int(target in {cand['primaryNorm'], cand['originalNorm']})
            fuzz_score = max(
                fuzz.token_set_ratio(cand['primaryNorm'], target) if cand['primaryNorm'] else 0,
                fuzz.token_set_ratio(cand['originalNorm'], target) if cand['originalNorm'] else 0,
            )
            year = cand.get('startYear')
            recent_bias = 0
            if year is not None:
                if year >= current_year - 2:
                    recent_bias = 2
                elif year >= current_year - 10:
                    recent_bias = 1
            score_key = (
                exact,
                fuzz_score,
                1 if cand.get('titleType') == 'movie' else 0,
                recent_bias,
                votes,
                year or 0,
            )
            if chosen is None or score_key > chosen_key:
                chosen = (rating_val, cand.get('runtimeMinutes'), f"https://www.imdb.com/title/{cand['tconst']}/")
                chosen_key = score_key

        if chosen_key is not None and (chosen_key[0] == 1 or chosen_key[1] >= 90):
            out[title] = chosen
        else:
            out[title] = (None, None, None)
    return out


# =========================
# REGEX / CONSTANTS
# =========================
SHOWTIME_HREF_RE = re.compile(
    r"""(?:https?://(?:www\.)?amctheatres\.com)?/showtimes(?:/all/[^\s"'?#]+/[^\s"'?#]+/[^\s"'?#]+)?/\d+""",
    re.I,
)
MOVIE_HREF_RE    = re.compile(r"(?:https?://(?:www\.)?amctheatres\.com)?/movies/", re.I)
TIME_RE          = re.compile(r"(\d{1,2}:\d{2}\s*[ap]m)", re.I)
AMC_RUNTIME_RE   = re.compile(r"(\d+)\s*hr?\s*(\d+)\s*min|(\d+)\s*min", re.I)

# Text fallbacks (handle normal % and fullwidth ％)
RT_TOMA_TXT_1 = re.compile(r"(\d{1,3})\s*[％%]\s*Tomatometer", re.I)
RT_TOMA_TXT_2 = re.compile(r"Tomatometer\s*(\d{1,3})\s*[％%]", re.I)

RT_AUD_TXT_1  = re.compile(r"(\d{1,3})\s*[％%]\s*(Popcornmeter|Audience\s*Score)", re.I)
RT_AUD_TXT_2  = re.compile(r"(Popcornmeter|Audience\s*Score)\s*(\d{1,3})\s*[％%]", re.I)

# JSON-ish fallbacks
RT_TOMA_JSON_1 = re.compile(r'"tomatometerScore"\s*:\s*(\d{1,3})', re.I)
RT_TOMA_JSON_2 = re.compile(r'"tomatometerScore"\s*:\s*\{[^}]{0,300}?"value"\s*:\s*(\d{1,3})', re.I)

RT_AUD_JSON_1  = re.compile(r'"audienceScore"\s*:\s*(\d{1,3})', re.I)
RT_AUD_JSON_2  = re.compile(r'"audienceScore"\s*:\s*\{[^}]{0,300}?"value"\s*:\s*(\d{1,3})', re.I)
RT_AUD_JSON_3  = re.compile(r'"popcornmeter"\s*:\s*(\d{1,3})', re.I)
RT_AUD_JSON_4  = re.compile(r'"popcornmeter"\s*:\s*\{[^}]{0,300}?"score"\s*:\s*(\d{1,3})', re.I)


# =========================
# TIME / SESSION HELPERS
# =========================
def upcoming_weekend_pacific() -> Tuple[date, date]:
    """Return upcoming Saturday/Sunday in America/Los_Angeles."""
    try:
        from zoneinfo import ZoneInfo
        today = datetime.now(ZoneInfo("America/Los_Angeles")).date()
    except Exception:
        today = date.today()

    if OVERRIDE_SATURDAY:
        sat = datetime.strptime(OVERRIDE_SATURDAY, "%Y-%m-%d").date()
        return sat, sat + timedelta(days=1)

    wd = today.weekday()  # Mon=0 .. Sun=6
    sat = today + timedelta(days=(5 - wd)) if wd <= 5 else today + timedelta(days=6)
    return sat, sat + timedelta(days=1)

def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update({
        "User-Agent": (
            "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
            "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        ),
        "Accept-Language": "en-US,en;q=0.9",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Referer": "https://www.google.com/",
    })
    return s

def fetch_html(session: requests.Session, url: str, params: dict | None = None, tries: int = 3) -> Tuple[int, str]:
    """Return (status_code, text). Retries a few times."""
    last_status, last_text = 0, ""
    for i in range(tries):
        try:
            r = session.get(url, params=params, timeout=30, allow_redirects=True)
            last_status = r.status_code
            last_text = r.text or ""
            if last_status == 200 and "Access Denied" not in last_text:
                return last_status, last_text
        except Exception:
            pass
        time.sleep(1.0 * (i + 1))
    return last_status, last_text


def looks_like_amc_interstitial(html_txt: str, final_url: str = "") -> bool:
    low = (html_txt or "").lower()
    final_low = (final_url or "").lower()
    markers = (
        "the site requires javascript to be enabled",
        "queue.amctheatres.com",
        "global safety net",
        "enable-javascript.com",
    )
    return any(m in low for m in markers) or "queue.amctheatres.com" in final_low


def fetch_html_with_browser(url: str, params: dict | None = None, timeout_ms: int = 45000) -> Tuple[int, str]:
    """Use a real browser for AMC pages that now sit behind a JS/queue interstitial."""
    try:
        from playwright.sync_api import sync_playwright
    except Exception as e:
        raise RuntimeError(
            "Playwright is required for AMC browser fallback. Add 'playwright' to requirements.txt and run "
            "'python -m playwright install --with-deps chromium' in CI."
        ) from e

    full_url = requests.Request("GET", url, params=params).prepare().url

    with sync_playwright() as p:
        browser = p.chromium.launch(
            headless=True,
            args=[
                "--disable-blink-features=AutomationControlled",
                "--no-sandbox",
            ],
        )
        context = browser.new_context(
            user_agent=(
                "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                "(KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36"
            ),
            locale="en-US",
            timezone_id="America/Los_Angeles",
            viewport={"width": 1440, "height": 2200},
            java_script_enabled=True,
        )

        page = context.new_page()
        response = page.goto(full_url, wait_until="domcontentloaded", timeout=timeout_ms)

        # Give AMC's queue / JS challenge a few seconds to complete if present.
        for _ in range(3):
            try:
                page.wait_for_load_state("networkidle", timeout=6000)
            except Exception:
                pass
            page.wait_for_timeout(2500)

            html_txt = page.content()
            if not looks_like_amc_interstitial(html_txt, page.url):
                break

            try:
                page.reload(wait_until="domcontentloaded", timeout=timeout_ms)
            except Exception:
                pass

        html_txt = page.content()
        status = response.status if response is not None else 200

        context.close()
        browser.close()
        return status, html_txt


def fetch_amc_html(session: requests.Session, url: str, params: dict | None = None) -> Tuple[int, str]:
    status, html_txt = fetch_html(session, url, params=params, tries=2)
    if status == 200 and not looks_like_amc_interstitial(html_txt):
        return status, html_txt

    target = requests.Request("GET", url, params=params).prepare().url
    print(f"[INFO] AMC returned a JS/queue page for {target}; retrying with Playwright browser fallback...")

    try:
        status2, html_txt2 = fetch_html_with_browser(url, params=params)
        if status2 == 200 and not looks_like_amc_interstitial(html_txt2):
            return status2, html_txt2
        return status2, html_txt2
    except Exception as e:
        print(f"[WARN] Browser fallback failed for {target}: {e}")
        return status, html_txt


# =========================
# AMC SCRAPING
# =========================
def extract_time(txt: str) -> Optional[str]:
    m = TIME_RE.search(txt or "")
    if not m:
        return None
    return re.sub(r"\s+", " ", m.group(1).strip().lower())

def parse_time_for_sort(t: str) -> int:
    try:
        dt = datetime.strptime(t.strip().upper(), "%I:%M %p")
        return dt.hour * 60 + dt.minute
    except Exception:
        return 10**9

def is_a_list_excluded_near_tag(tag: Tag, max_parent_levels: int = 6) -> bool:
    """
    More reliable AMC parsing: only look within a limited number of ancestor levels
    so we don't accidentally match page-wide legends/footers.
    """
    lvl = 0
    for p in tag.parents:
        if not isinstance(p, Tag):
            continue
        txt = p.get_text(" ", strip=True).lower()
        if "excluded from a-list" in txt:
            return True
        lvl += 1
        if lvl >= max_parent_levels:
            break
    return False

def looks_like_format_label(txt: str) -> bool:
    t = (txt or "").strip()
    if not t or len(t) > 140:
        return False
    needles = ["imax", "dolby", "prime", "reald", "laser", "fan faves", "no trailers", "spoken", "subtitles", "open caption", "caption"]
    tl = t.lower()
    return any(n in tl for n in needles)


def href_matches_movie(href: str) -> bool:
    return bool(MOVIE_HREF_RE.search((href or "").strip()))

def href_matches_showtime(href: str) -> bool:
    return bool(SHOWTIME_HREF_RE.search((href or "").strip()))

def _nearest_movie_container(tag: Tag) -> Tag:
    for p in [tag, *tag.parents]:
        if not isinstance(p, Tag):
            continue
        if p.name in {"article", "section", "li"}:
            return p
        classes = " ".join(p.get("class", [])).lower()
        if any(k in classes for k in ["movie", "showtimes", "grid", "card", "tile", "session"]):
            return p
    return tag

def _collect_movie_blocks(soup: BeautifulSoup) -> List[Tuple[Tag, str]]:
    blocks: List[Tuple[Tag, str]] = []
    seen = set()

    # First preference: heading tags that contain a movie link.
    for el in soup.find_all(["h1", "h2", "h3", "h4", "h5"]):
        a = el.find("a", href=True)
        if not a or not href_matches_movie(a.get("href", "")):
            continue
        title = a.get_text(" ", strip=True)
        key = (title.lower(), getattr(el, "sourceline", None))
        if title and key not in seen:
            blocks.append((el, title))
            seen.add(key)

    if blocks:
        return blocks

    # Fallback: any movie link, using a nearby semantic container if available.
    seen_titles = set()
    for a in soup.find_all("a", href=True):
        href = a.get("href", "")
        if not href_matches_movie(href):
            continue
        title = a.get_text(" ", strip=True)
        if not title:
            continue
        norm = re.sub(r"\s+", " ", title.strip().lower())
        if norm in seen_titles:
            continue
        blocks.append((_nearest_movie_container(a), title))
        seen_titles.add(norm)

    return blocks

def _likely_showtime_tag(el: Tag, txt: str) -> bool:
    if not txt or not TIME_RE.search(txt):
        return False
    clean = re.sub(r"\s+", " ", txt.strip())
    if len(clean) > 120:
        return False

    if el.name == "a" and href_matches_showtime(el.get("href", "")):
        return True

    classes = " ".join(el.get("class", [])).lower()
    attrs = " ".join(f"{k}={v}" for k, v in getattr(el, "attrs", {}).items()).lower()
    if el.name in {"button", "a"}:
        return True
    if any(k in classes for k in ["showtime", "session", "time"]):
        return True
    if any(k in attrs for k in ["showtime", "session", "time"]):
        return True

    # A very small element that is basically just a time label can still be a showtime pill.
    stripped = clean.lower()
    stripped = re.sub(r"up to \d+% off\.?","", stripped).strip()
    return bool(TIME_RE.fullmatch(stripped) or TIME_RE.match(stripped))

def scrape_amc_showtimes_for_date(session: requests.Session, theatre_name: str, showtimes_url: str, d: date) -> List[dict]:
    status, html_txt = fetch_amc_html(session, showtimes_url, params={"date": d.isoformat()})
    if status != 200:
        return []

    soup = BeautifulSoup(html_txt, "html.parser")
    movie_blocks = _collect_movie_blocks(soup)

    out: List[dict] = []
    seen = set()

    for idx, (block, title) in enumerate(movie_blocks):
        stop_tag = movie_blocks[idx + 1][0] if idx + 1 < len(movie_blocks) else None
        current_format, current_runtime = None, None
        el = block.next_element

        while el is not None and el is not stop_tag:
            if isinstance(el, Tag):
                txt = re.sub(r"\s+", " ", el.get_text(" ", strip=True))

                if looks_like_format_label(txt):
                    current_format = txt

                rt_m = AMC_RUNTIME_RE.search(txt)
                if rt_m and current_runtime is None:
                    if rt_m.group(1):
                        current_runtime = int(rt_m.group(1)) * 60 + int(rt_m.group(2))
                    else:
                        current_runtime = int(rt_m.group(3))

                if _likely_showtime_tag(el, txt):
                    time_m = TIME_RE.search(txt)
                    if time_m:
                        is_excluded = is_a_list_excluded_near_tag(el)
                        row = {
                            "movie_title": title,
                            "theatre": theatre_name,
                            "show_date": d.isoformat(),
                            "show_time": time_m.group(1).lower(),
                            "format_label": current_format,
                            "runtime_min": current_runtime,
                            "a_list_excluded": is_excluded,
                        }
                        dedupe_key = (
                            row["movie_title"].strip().lower(),
                            row["theatre"].strip().lower(),
                            row["show_date"],
                            row["show_time"],
                            (row["format_label"] or "").strip().lower(),
                        )
                        if dedupe_key not in seen:
                            out.append(row)
                            seen.add(dedupe_key)
            el = el.next_element

    # Fail fresh instead of silently using old data; this scraper should only succeed on the requested AMC date.
    out = [r for r in out if r.get("show_date") == d.isoformat()]
    return out


# =========================
# TITLE NORMALIZATION
# =========================
def candidate_title_variants(title: str) -> List[str]:
    t = re.sub(r"\s+", " ", (title or "").strip())
    if not t:
        return []
    variants = [t]
    for pat in [
        r"\s+\d{1,3}(st|nd|rd|th)\s+anniversary\s*$",
        r"\s+\d{1,3}th\s+anniversary\s*$",
        r"\s+early\s+access(\s+event)?\s*$",
        r"\s+sneak\s+peek\s*$",
        r"\s+fan\s+event\s*$",
        r"\s+re-?release\s*$",
        r"\s+opening\s+night\s*$",
        r"\s+q&a\s*$",
        r"\s+double\s+feature\s*$",
        r"\s+live\s+event\s*$",
    ]:
        c = re.sub(pat, "", t, flags=re.I).strip(" -:")
        if c and c.lower() != t.lower():
            variants.append(c)

    extra = []
    for v in variants:
        c = re.sub(r":\s+the\s+imax\s+experience$", "", v, flags=re.I).strip(" -:")
        if c and c.lower() != v.lower():
            extra.append(c)
        c = re.sub(r"\s*\(.*?\)", "", v).strip(" -:")
        if c and c.lower() != v.lower():
            extra.append(c)
    variants.extend(extra)

    seen, out = set(), []
    for v in variants:
        v = re.sub(r"\s+", " ", v).strip()
        k = v.lower()
        if v and k not in seen:
            seen.add(k)
            out.append(v)
    return out


# =========================
# ROTTEN TOMATOES (SLUG-FIRST, ROBUST PARSE)
# =========================
def rt_slugify(title: str) -> str:
    s = (title or "").lower().strip()
    s = re.sub(r"['’]", "", s)          # drop apostrophes
    s = re.sub(r"[^a-z0-9]+", "_", s)   # non-alnum -> underscore
    s = re.sub(r"_+", "_", s).strip("_")
    return s

def _int0_100(x: Optional[str]) -> Optional[int]:
    if not x:
        return None
    x = str(x).strip()
    if not re.fullmatch(r"\d{1,3}", x):
        return None
    n = int(x)
    if 0 <= n <= 100:
        return n
    return None

def rt_parse_scores(decoded_html: str, soup: BeautifulSoup) -> Tuple[Optional[int], Optional[int]]:
    """
    Try (1) component attributes (most reliable), (2) embedded JSON, (3) visible text.
    Returns (audience, critic).
    """
    audience = None
    critic = None

    # (1) Component attributes (RT often renders <score-board ... tomatometerscore=".." audiencescore="..">)
    attr_candidates = []
    for attr in ["tomatometerscore", "tomatometerScore", "tomatometerscoreallcritics", "tomatometerscoreall"]:
        tag = soup.find(attrs={attr: True})
        if tag:
            attr_candidates.append((attr, tag.get(attr)))

    for attr, val in attr_candidates:
        n = _int0_100(val)
        if n is not None:
            critic = n
            break

    aud_attr_candidates = []
    for attr in ["audiencescore", "audienceScore", "popcornmeterscore", "popcornmeterScore"]:
        tag = soup.find(attrs={attr: True})
        if tag:
            aud_attr_candidates.append((attr, tag.get(attr)))

    for attr, val in aud_attr_candidates:
        n = _int0_100(val)
        if n is not None:
            audience = n
            break

    if critic is not None and audience is not None:
        return audience, critic

    # (2) Embedded JSON patterns in the HTML
    if critic is None:
        for rx in [RT_TOMA_JSON_2, RT_TOMA_JSON_1]:
            m = rx.search(decoded_html)
            if m:
                critic = _int0_100(m.group(1))
                if critic is not None:
                    break

    if audience is None:
        for rx in [RT_AUD_JSON_4, RT_AUD_JSON_3, RT_AUD_JSON_2, RT_AUD_JSON_1]:
            m = rx.search(decoded_html)
            if m:
                audience = _int0_100(m.group(1))
                if audience is not None:
                    break

    if critic is not None and audience is not None:
        return audience, critic

    # (3) Visible text fallback
    full_text = soup.get_text(" ", strip=True)
    full_text = re.sub(r"\s+", " ", full_text)

    # try to anchor around H1
    h1 = soup.find("h1")
    tail = full_text
    if h1:
        anchor = h1.get_text(" ", strip=True)
        if anchor:
            pos = full_text.lower().find(anchor.lower())
            if pos != -1:
                tail = full_text[pos:pos + 50000]

    if critic is None:
        m = (RT_TOMA_TXT_1.search(tail) or RT_TOMA_TXT_2.search(tail) or
             RT_TOMA_TXT_1.search(full_text) or RT_TOMA_TXT_2.search(full_text))
        if m:
            critic = _int0_100(m.group(1))

    if audience is None:
        m = RT_AUD_TXT_1.search(tail)
        if m:
            audience = _int0_100(m.group(1))
        else:
            m = RT_AUD_TXT_2.search(tail)
            if m:
                audience = _int0_100(m.group(2))
            else:
                m = RT_AUD_TXT_1.search(full_text)
                if m:
                    audience = _int0_100(m.group(1))
                else:
                    m = RT_AUD_TXT_2.search(full_text)
                    if m:
                        audience = _int0_100(m.group(2))

    return audience, critic

def rt_get_scores(
    session: requests.Session,
    title: str,
    cache: Dict[str, Tuple[Optional[int], Optional[int], Optional[str]]],
    debug: bool = False) -> Tuple[Optional[int], Optional[int], Optional[str]]:

    key = (title or "").lower().strip()
    if key in cache:
        return cache[key]

    # Get relevant years based on current date (2026)
    current_year = date.today().year
    years_to_check = [current_year, current_year + 1, current_year - 1]

    for q in candidate_title_variants(title):
        base_slug = rt_slugify(q)
        if not base_slug:
            continue

        # Create a list of URL attempts: [slug_2026, slug_2027, slug_2025, slug]
        url_attempts = [f"{RT_BASE}/m/{base_slug}_{y}" for y in years_to_check]
        url_attempts.append(f"{RT_BASE}/m/{base_slug}")

        for rt_url in url_attempts:
            status, raw_html = fetch_html(session, rt_url, params=None, tries=2)

            if debug:
                print(f"[RT] Testing {rt_url} -> Status: {status}")

            if status != 200 or not raw_html:
                continue

            # Double-check we didn't land on a '404' or 'Page Not Found'
            # RT often returns a 200 but shows a "sorry" page
            lower_html = raw_html.lower()
            if "404 - page not found" in lower_html or "couldn't find the page you're looking for" in lower_html:
                continue

            decoded = html_lib.unescape(raw_html)
            soup = BeautifulSoup(decoded, "html.parser")

            # Validate the title on the page matches our intent to avoid
            # false positives (e.g., scraping "Wicked" (1998) for "Wicked" (2024))
            # We look for the <h1> tag or the title meta tag
            # page_title = soup.find("h1")
            # page_title_text = page_title.get_text(strip=True).lower() if page_title else ""

            # Use fuzzy matching or simple inclusion to verify
            # if base_slug.replace("_", " ") not in page_title_text:
            #     if debug:
            #         print(f"[RT] Skipping {rt_url}: Title mismatch (Page says: {page_title_text})")
            #     continue


            aud, crit = rt_parse_scores(decoded, soup)
            cache[key] = (aud, crit, rt_url)
            return cache[key]

    cache[key] = (None, None, None)
    return cache[key]


# =========================
# IMDB (rating + runtime + canonical URL)
# =========================
def imdb_rating_runtime_url(
    session: requests.Session,
    title: str,
    cache: Dict[str, Tuple[Optional[float], Optional[int], Optional[str]]]
) -> Tuple[Optional[float], Optional[int], Optional[str]]:
    return cache.get((title or '').strip(), (None, None, None))


# =========================
# SORTING RULE (YOUR SPEC)
#   0) RT audience
#   1) IMDb (if no RT audience)
#   2) RT critic (if neither)
# =========================
def pick_primary(rt_aud: Optional[int], imdb: Optional[float], rt_crit: Optional[int]) -> Tuple[int, Optional[float], str]:
    """
    Returns (rank, score, source)
    rank: 0 audience, 1 imdb, 2 critic, 3 none
    score: audience / imdb*10 / critic
    """
    if rt_aud is not None:
        return 0, float(rt_aud), "RT_AUDIENCE"
    if imdb is not None:
        return 1, float(imdb) * 10.0, "IMDB"
    if rt_crit is not None:
        return 2, float(rt_crit), "RT_CRITIC"
    return 3, None, "NONE"


# =========================
# OUTPUT FORMATTING
# =========================
def short_fmt(s: Optional[str]) -> Optional[str]:
    # Check if 's' is None, or if it's a float (like NaN)
    if not isinstance(s, str):
        return None

    s = re.sub(r"\s+", " ", s).strip()
    return s if len(s) <= 40 else s[:37] + "..."

def build_showtimes_cell(df_st: pd.DataFrame) -> str:
    lines = []
    for theatre in sorted(df_st["theatre"].unique()):
        df_th = df_st[df_st["theatre"] == theatre].copy()

        lines.append(f"{theatre}")
        for show_date in sorted(df_th["show_date"].unique()):
            df_d = df_th[df_th["show_date"] == show_date].copy()
            df_d["t_sort"] = df_d["show_time"].apply(parse_time_for_sort)
            df_d = df_d.sort_values("t_sort")

            times = []
            for _, r in df_d.iterrows():
                excl = "⛔" if bool(r["a_list_excluded"]) else ""
                fmt = short_fmt(r.get("format_label"))
                fmt_txt = f" [{fmt}]" if fmt else ""
                times.append(f"{r['show_time']}{excl}{fmt_txt}")

            lines.append(f"• {show_date}: " + ", ".join(times))

    return "\n".join(lines)

# =========================
# RUN
# =========================
sat, sun = upcoming_weekend_pacific()
dates = [sat, sun]

session = make_session()

# Scrape showtimes
showtimes: List[dict] = []
for th in THEATRES:
    for d in dates:
        try:
            showtimes.extend(scrape_amc_showtimes_for_date(session, th["name"], th["url"], d))
            time.sleep(0.2)
        except Exception as e:
            print(f"[WARN] Failed scraping {th['name']} {d}: {e}")

if not showtimes:
    raise RuntimeError(
        "No fresh AMC showtimes were parsed for the requested dates. "
        "This run does not fall back to older data. "
        "AMC likely changed the page markup again or blocked the browser session."
    )

df_show = pd.DataFrame(showtimes)
# Add this line right after creating df_show:
df_show["format_label"] = df_show["format_label"].fillna("")

# Lookup scores (1 row per movie)
movie_titles = sorted(df_show["movie_title"].unique())
imdb_cache: Dict[str, Tuple[Optional[float], Optional[int], Optional[str]]] = build_imdb_lookup(session, movie_titles)
rt_cache: Dict[str, Tuple[Optional[int], Optional[int], Optional[str]]] = {}

movie_rows = []
for title in movie_titles:
    rt_aud, rt_crit, rt_url = rt_get_scores(session, title, rt_cache, debug=DEBUG_RT)
    imdb, imdb_runtime_min, imdb_url = imdb_rating_runtime_url(session, title, imdb_cache)

    rank, prim_score, prim_src = pick_primary(rt_aud, imdb, rt_crit)

    # Priority: AMC runtime first, IMDb runtime second
    amc_rt = df_show[df_show["movie_title"] == title]["runtime_min"].dropna()
    rt_min = int(amc_rt.iloc[0]) if not amc_rt.empty else imdb_runtime_min

    h, m = divmod(rt_min, 60) if rt_min else (0, 0)
    fmt_rt = f"{h}h {m}m" if h else (f"{m}m" if m else None)

    movie_rows.append({
        "movie_title": title,
        "runtime": fmt_rt,
        "runtime_min": rt_min,  # hidden helper
        "rt_audience": rt_aud,
        "imdb_rating": imdb,
        "rt_critic": rt_crit,
        "sort_rank": rank,
        "primary_score": prim_score,
        "primary_source": prim_src,
        "rt_url": rt_url,
        "imdb_url": imdb_url,
    })
    time.sleep(0.2)

df_movies = pd.DataFrame(movie_rows)

# Aggregate showtimes into one cell per movie + a_list_excluded_any (ANY showtime excluded)
agg = []
for title, df_st in df_show.groupby("movie_title"):
    agg.append({
        "movie_title": title,
        "a_list_excluded_any": bool(df_st["a_list_excluded"].fillna(False).any()),
        "showtimes": build_showtimes_cell(df_st),
    })
df_agg = pd.DataFrame(agg)

df_summary = (
    df_movies.merge(df_agg, on="movie_title", how="left")
            .sort_values(
                by=["sort_rank", "primary_score", "movie_title"],
                ascending=[True, False, True]
            )
            .reset_index(drop=True)
)

# Keep raw numeric columns and hidden URLs; postprocess_report.py will render score links.
df_display = df_summary.copy()


def fmt_imdb(x):
    if pd.isna(x):
        return ""
    try:
        return round(float(x), 1)
    except Exception:
        return _clean_missing_text(x)


df_display["imdb_rating"] = df_display["imdb_rating"].apply(fmt_imdb)

# Keep URL helper columns for the HTML postprocessor.
desired = [
    "movie_title",
    "runtime",
    "rt_critic",
    "rt_audience",
    "imdb_rating",
    "showtimes",
    "rt_url",
    "imdb_url",
]
df_display = df_display[[c for c in desired if c in df_display.columns]]

pd.set_option("display.max_colwidth", None)

print(f"Upcoming weekend (Pacific): {sat.isoformat()} (Sat), {sun.isoformat()} (Sun)")
html_out = df_display.to_html(index=False, escape=False).replace("\\n", "<br>")

css = """
<style>
table.dataframe th, table.dataframe td {
  text-align: left !important;
  vertical-align: top;
}
</style>
"""

display(HTML(css + html_out))



Upcoming weekend (Pacific): 2026-02-14 (Sat), 2026-02-15 (Sun)


movie_title,runtime,rt_audience,imdb_rating,rt_critic,showtimes
Stray Kids : The dominATE Experience,2h 26m,100.0,None,NaN,"AMC Orange 30• 2026-02-14: 10:35 pm [Laser at AMC]• 2026-02-15: 10:30 am [Laser at AMC], 2:15 pm [Laser at AMC], 6:00 pm [Laser at AMC], 9:45 pm [Laser at AMC]AMC Tustin 14 @ The District• 2026-02-15: 12:40 pm [Laser at AMC]"
The Rose: Come Back to Me,1h 37m,100.0,None,100.0,AMC Orange 30• 2026-02-15: 4:00 pm⛔ [Laser at AMC]
Time Hoppers: The Silk Road,1h 30m,99.0,None,NaN,AMC Orange 30• 2026-02-15: 8:45 am⛔ [Laser at AMC]
Melania,1h 44m,98.0,None,11.0,AMC Tustin 14 @ The District• 2026-02-15: 4:50 pm [Laser at AMC]
Nirvanna the Band the Show the Movie,1h 42m,98.0,None,97.0,"AMC Orange 30• 2026-02-14: 11:00 pm [Laser at AMC]• 2026-02-15: 11:00 am [Laser at AMC], 2:00 pm [Laser at AMC], 5:00 pm [Laser at AMC], 8:00 pm [Laser at AMC], 11:00 pm [Laser at AMC]AMC Tustin 14 @ The District• 2026-02-14: 10:05 pm [Laser at AMC]• 2026-02-15: 10:55 am [Laser at AMC], 1:40 pm [Laser at AMC], 4:30 pm [Laser at AMC], 7:20 pm [Laser at AMC], 10:10 pm [Laser at AMC]"
Sinners,2h 17m,96.0,None,97.0,AMC Orange 30• 2026-02-15: 11:45 am [Laser at AMC]
Solo Mio,1h 40m,96.0,None,80.0,"AMC Orange 30• 2026-02-14: 10:05 pm [Laser at AMC]• 2026-02-15: 10:00 am [Laser at AMC], 1:00 pm [Laser at AMC], 8:00 pm [Laser at AMC], 10:45 pm [Laser at AMC]AMC Tustin 14 @ The District• 2026-02-14: 10:10 pm [Laser at AMC]• 2026-02-15: 9:45 am [Laser at AMC], 4:20 pm [Laser at AMC], 8:05 pm [Laser at AMC]"
Zootopia 2,1h 47m,96.0,None,91.0,"AMC Orange 30• 2026-02-15: 10:05 am [Laser at AMC], 12:55 pm [Laser at AMC], 3:45 pm [Laser at AMC], 7:30 pm [Laser at AMC], 10:30 pm [Laser at AMC]AMC Tustin 14 @ The District• 2026-02-15: 10:40 am [Laser at AMC], 1:50 pm [Laser at AMC], 4:00 pm [Laser at AMC]AMC Woodbridge 5• 2026-02-15: 10:00 am [Laser at AMC], 12:45 pm [Laser at AMC], 3:30 pm [Laser at AMC], 6:15 pm [Laser at AMC]"
Goat,1h 39m,93.0,None,79.0,"AMC Orange 30• 2026-02-14: 10:45 pm [Laser at AMC]• 2026-02-15: 8:45 am [Laser at AMC], 9:45 am [Dolby Cinema at AMC], 10:30 am [Laser at AMC], 11:30 am [Laser at AMC], 12:30 pm [Dolby Cinema at AMC], 1:15 pm [Laser at AMC], 2:15 pm [Laser at AMC], 4:00 pm [Laser at AMC], 5:00 pm [Laser at AMC], 8:00 pm [Laser at AMC], 10:45 pm [Laser at AMC]AMC Tustin 14 @ The District• 2026-02-15: 10:00 am [Laser at AMC], 11:00 am [Laser at AMC], 1:00 pm [Dolby Cinema at AMC], 1:45 pm [Laser at AMC], 3:45 pm [Dolby Cinema at AMC], 5:00 pm [Laser at AMC], 6:15 pm [Laser at AMC], 9:00 pm [Laser at AMC]AMC Woodbridge 5• 2026-02-15: 10:00 am [Laser at AMC], 10:30 am [Laser at AMC], 11:00 am [Laser at AMC], 1:20 pm [Laser at AMC], 4:25 pm [Laser at AMC], 6:45 pm [Laser at AMC], 9:00 pm [Laser at AMC]"
Hamnet,2h 5m,93.0,None,86.0,"AMC Orange 30• 2026-02-15: 2:05 pm [Laser at AMC], 7:40 pm [Laser at AMC]"
